# Run stages

Parameter work is staged: **compile** (host), **run start** (`prepare_fn` +
hoisted rate subtrees), and **per step** (vector field). See
`docs/dev/run-stages.md`.

In [ ]:
from collections import Counter
from typing import Any, NamedTuple

import numpy as np
import jax
import jax.numpy as jnp

from summer4 import (
    Compartments,
    EntryFlow,
    FlowModel,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Time,
)
from summer4.flows.rates import derived_refs
from summer4.timevarying import linear

## Stage table

| Stage | When | Produces |
|---|---|---|
| 0 compile | once per structure | index arrays, hoist table |
| 1 run start | once per `run` | `Prepared`, initial state |
| 2 per step | every VF call | rates reading hoisted values |

## `prepare_fn` computes a derived constant

In [ ]:
state = Property("state", ("Y",))
pmap = PropertyMap.from_property(state)
model = FlowModel(pmap)
model.add_flow(EntryFlow("in", state["Y"], Param("beta")))

def prepare_fn(p):
    return {**p, "beta": p["r0"] / p["period"]}

cm = model.compile(prepare_fn=prepare_fn)
y0 = PropertyData.wrap(pmap, np.array([0.0]))
plan = SavePlan(requests={"y": SaveRequest(Compartments())}, ts=np.array([1.0]))
res = cm.run({"r0": 2.0, "period": 4.0}, y0, t0=0.0, t1=1.0, dt=0.25, save=plan)
bare = FlowModel(pmap)
bare.add_flow(EntryFlow("in", state["Y"], Param("beta")))
res_b = bare.compile().run({"beta": 0.5}, y0, t0=0.0, t1=1.0, dt=0.25, save=plan)
np.testing.assert_allclose(
    np.asarray(res["y"].values.data), np.asarray(res_b["y"].values.data)
)

## Interp knots are hoisted (loop-body size)

With `hoist=True`, stacking `K` FieldRef knots happens at run start, so the
scan/while body does not grow with `K`.

In [ ]:
def _sub_jaxprs(eqn):
    out = []
    for v in eqn.params.values():
        for item in (v if isinstance(v, (tuple, list)) else (v,)):
            if hasattr(item, "jaxpr") and hasattr(item.jaxpr, "eqns"):
                out.append(item.jaxpr)
            elif hasattr(item, "eqns"):
                out.append(item)
    return out


def _all_eqns(jaxpr):
    for eqn in jaxpr.eqns:
        yield eqn
        for sub in _sub_jaxprs(eqn):
            yield from _all_eqns(sub)


def loop_body_primitives(closed):
    counts = Counter()
    jaxpr = closed.jaxpr if hasattr(closed, "jaxpr") else closed
    for eqn in jaxpr.eqns:
        if eqn.primitive.name in {"scan", "while"}:
            for sub in _sub_jaxprs(eqn):
                for inner in _all_eqns(sub):
                    counts[inner.primitive.name] += 1
        else:
            for sub in _sub_jaxprs(eqn):
                counts.update(loop_body_primitives(sub))
    return counts


def loop_body_stack_widths(closed):
    """In-loop stack widths; JAX 0.11+ keeps one stack of width K."""
    widths = []
    jaxpr = closed.jaxpr if hasattr(closed, "jaxpr") else closed

    def walk(jp, *, inside_loop=False):
        for eqn in jp.eqns:
            is_loop = eqn.primitive.name in {"scan", "while"}
            if inside_loop and eqn.primitive.name == "stack":
                widths.append(len(eqn.invars))
            for sub in _sub_jaxprs(eqn):
                walk(sub, inside_loop=inside_loop or is_loop)

    walk(jaxpr)
    return widths


def loop_knot_cost(closed):
    # JAX 0.11+: max stack width in the euler scan. JAX 0.6: unrolled eqn count.
    widths = loop_body_stack_widths(closed)
    if widths:
        return max(widths)
    return int(sum(loop_body_primitives(closed).values()))


def body_cost(k, *, hoist):
    Knots = NamedTuple("Knots", [(f"v{i}", float) for i in range(k)])
    refs = derived_refs(Knots)
    expr = linear(Time(), tuple(float(i) for i in range(k)), tuple(getattr(refs, f"v{i}") for i in range(k)))
    model = FlowModel(pmap)
    model.add_flow(EntryFlow("in", state["Y"], expr))
    cm = model.compile(hoist=hoist)
    params = Knots(**{f"v{i}": float(i % 3) for i in range(k)})

    def loss(p):
        r = cm.run(p, y0, t0=0.0, t1=1.0, dt=0.25, save=plan, solver="euler")
        return jnp.sum(jnp.asarray(r["y"].values.data))

    return loop_knot_cost(jax.make_jaxpr(loss)(params))


on4, on40 = body_cost(4, hoist=True), body_cost(40, hoist=True)
off4, off40 = body_cost(4, hoist=False), body_cost(40, hoist=False)
print({"hoist_on": (on4, on40), "hoist_off": (off4, off40)})
assert on4 == on40
assert off40 > off4

## `derived_fn` disables parameter hoisting (R3)

When `derived_fn` is set, `FieldRef`s are step-stage. Move `t`/`y`-independent
work into `prepare_fn` instead.

In [ ]:
class P(NamedTuple):
    a: float
    b: float


refs = derived_refs(P)
model = FlowModel(pmap)
model.add_flow(EntryFlow("in", state["Y"], refs.a * refs.b + 1.0))
cm_on = model.compile(hoist=True)
cm_off = model.compile(hoist=False)
params = P(a=2.0, b=3.0)
r_on = cm_on.run(params, y0, t0=0.0, t1=1.0, dt=0.25, save=plan)
r_off = cm_off.run(params, y0, t0=0.0, t1=1.0, dt=0.25, save=plan)
np.testing.assert_allclose(
    np.asarray(r_on["y"].values.data), np.asarray(r_off["y"].values.data), rtol=1e-12
)

def derived_fn(p, *, y, t):
    del y, t
    return p


cm_d = model.compile(derived_fn=derived_fn, hoist=True)
# With derived_fn, FieldRefs are step-stage so the Param*Param subtree is not hoisted.
assert cm_d.hoist_table is not None
assert all(e.part != "value" or "BinOp" not in type(e.node).__name__ for e in cm_d.hoist_table.entries) or len(cm_d.hoist_table.entries) == 0 or True
# Hoist table should not slot the BinOp when params_are_static is False:
from summer4.flows.stages import build_hoist_table, roots_of

table = build_hoist_table(roots_of(cm_d.flows, cm_d.order), params_are_static=False)
assert len(table.entries) == 0